## BIBLIOTECAS

### INSTALAÇÃO

In [ ]:
!pip install scikit_learn
!pip install pandas

### LIBS

In [ ]:
import pandas as pd
import unicodedata
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier

## MACHINE LEARNING

### SEPARAÇÃO DOS DADOS

In [ ]:
# Frases de pacientes
with open(rf"C:\Users\davih\OneDrive\Documentos\PROJETOS\Fiap\ANO 2\Cardio IA\config\Frases_simuladas.txt", 'r', encoding='utf-8') as f:
    frases_originais = [line.strip().replace('"', '') for line in f.readlines()]
display(frases_originais)

In [ ]:
# Dataframe de de sintomas
df_symptom = pd.read_csv(rf"C:\Users\davih\OneDrive\Documentos\PROJETOS\Fiap\ANO 2\Cardio IA\config\sintomas_doencas.csv")
df_symptom = df_symptom.dropna()
display(df_symptom)

### NORMALIZAÇÃO DOS DADOS

In [ ]:
# Função de limpeza (normaliza o texto do paciente e do CSV)
def limpar_texto(texto):
    if not isinstance(texto, str): return ""
    texto = "".join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'[^\w\s]', '', texto.lower())
    return texto.strip()

### RANDOM FOREST CLASSIFIER

In [ ]:
# Preparação do Conhecimento
df_symptom['texto_treino'] = (df_symptom['Sintoma 1'] + " " + df_symptom['Sintoma 2']).apply(limpar_texto)

# Vetorização com N-Grams (Capta expressões como 'dor no peito')
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words=['a', 'o', 'de', 'com', 'no', 'na'])
X_train = vectorizer.fit_transform(df_symptom['texto_treino'])
y_train = df_symptom['Doenca Associada']

# Treinamento com Random Forest (Modelo de Alta Performance)
modelo_rf = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_rf.fit(X_train, y_train)

# Diagnóstico Inteligente
frases_limpas = [limpar_texto(f) for f in frases_originais]
X_test = vectorizer.transform(frases_limpas)
previsoes = modelo_rf.predict(X_test)

# Tabela de Entrega
df_final = pd.DataFrame({
    'Relato do Paciente': frases_originais,
    'Diagnóstico (Random Forest)': previsoes
})

# Exportação
display(df_final)
df_final.to_csv('"C:\Users\davih\OneDrive\Documentos\PROJETOS\Fiap\ANO 2\Cardio IA\temp/Diagnosticos_RFC.csv', index=False, encoding='utf-8-sig')